# Kaggle Word2Vec NLP Tutorial Part 1：Bag-of-Words

这个 Notebook 复现 Kaggle `word2vec-nlp-tutorial` 的 Part 1。

虽然竞赛名字里有 Word2Vec，但这一部分做的是早期 NLP 的 Bag-of-Words 方法。它会把电影评论文本清洗成单词，再转换成词频向量，最后用随机森林判断评论是正面还是负面。

标签含义：

```text
1 = 正面评论
0 = 负面评论
```

在 Kaggle 上运行前，需要先在右侧 `Add data` 添加 `word2vec-nlp-tutorial` 竞赛数据。

## 1. 导入库

这里使用的库都是 Kaggle Notebook 常见预装库。

如果在本地运行，需要先安装：

```bash
pip install pandas numpy beautifulsoup4 scikit-learn
```

In [ ]:
import csv
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
print('Libraries imported successfully.')

## 2. 查找 Kaggle 数据路径

Kaggle Notebook 添加竞赛数据后，通常会出现在：

```text
/kaggle/input/word2vec-nlp-tutorial
```

下面的代码会自动尝试几个常见位置。如果你在本地运行，可以把数据放到 `data/` 目录。支持 `.tsv`，也支持 `.tsv.zip`。

In [ ]:
def show_available_inputs():
    input_root = Path('/kaggle/input')
    if input_root.exists():
        print('Available /kaggle/input entries:')
        for path in sorted(input_root.glob('*')):
            print(' -', path)
            for child in sorted(path.glob('*'))[:10]:
                print('    -', child.name)
    else:
        print('/kaggle/input does not exist. If you are running locally, put files under data/.')

def find_file(filename, search_dirs):
    for directory in search_dirs:
        directory = Path(directory)
        direct_path = directory / filename
        zipped_path = directory / f'{filename}.zip'
        if direct_path.exists():
            return direct_path
        if zipped_path.exists():
            return zipped_path
        if directory.exists():
            matches = list(directory.rglob(filename))
            if matches:
                return matches[0]
            zipped_matches = list(directory.rglob(f'{filename}.zip'))
            if zipped_matches:
                return zipped_matches[0]
    show_available_inputs()
    raise FileNotFoundError(
        f'Cannot find {filename}. In Kaggle, click Add Input / Add data and attach the word2vec-nlp-tutorial competition dataset.'
    )

search_dirs = [
    '/kaggle/input/word2vec-nlp-tutorial',
    '/kaggle/input',
    'data',
    '.',
]

train_path = find_file('labeledTrainData.tsv', search_dirs)
test_path = find_file('testData.tsv', search_dirs)

print('Train file:', train_path)
print('Test file: ', test_path)


## 3. 读取训练集和测试集

`labeledTrainData.tsv` 是训练集，有 `sentiment` 标签。

`testData.tsv` 是测试集，没有标签，我们要预测它的 `sentiment`。

In [ ]:
def read_tsv(path):
    path = Path(path)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as archive:
            tsv_names = [name for name in archive.namelist() if name.endswith('.tsv')]
            if not tsv_names:
                raise ValueError(f'No TSV file found inside {path}')
            with archive.open(tsv_names[0]) as file_obj:
                return pd.read_csv(file_obj, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)
    return pd.read_csv(path, header=0, delimiter='\t', quoting=csv.QUOTE_NONE)

train = read_tsv(train_path)
test = read_tsv(test_path)

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
display(train.head())

## 4. 文本清洗

早期 NLP 通常非常依赖文本清洗。

这里做几件事：

去掉 HTML 标签。  
只保留英文字母。  
统一转成小写。  
按空格切分单词。  
去掉停用词。

例如 `<br />`、标点符号、大小写混杂都会被处理掉。

In [ ]:
STOP_WORDS = set(ENGLISH_STOP_WORDS)

def review_to_words(raw_review):
    review_text = BeautifulSoup(raw_review, 'html.parser').get_text()
    letters_only = re.sub('[^a-zA-Z]', ' ', review_text)
    words = letters_only.lower().split()
    meaningful_words = [word for word in words if word not in STOP_WORDS]
    return ' '.join(meaningful_words)

print('Original review snippet:')
print(train['review'].iloc[0][:500])
print('\nCleaned review snippet:')
print(review_to_words(train['review'].iloc[0])[:500])

## 5. 清洗所有训练评论

每条训练评论都要用同一个清洗函数处理。

In [ ]:
clean_train_reviews = []

for i, review in enumerate(train['review'], start=1):
    if i % 1000 == 0:
        print(f'Cleaned {i} / {len(train)} training reviews')
    clean_train_reviews.append(review_to_words(review))

print('Number of cleaned training reviews:', len(clean_train_reviews))

## 6. Bag-of-Words 向量化

`CountVectorizer` 会建立词表，并统计每条评论里每个词出现了多少次。

这一步把文本变成机器学习模型能处理的数字矩阵。

`max_features=5000` 表示只保留最常见的 5000 个词。

In [ ]:
vectorizer = CountVectorizer(max_features=5000)
train_features = vectorizer.fit_transform(clean_train_reviews)

print('Train feature matrix shape:', train_features.shape)

可以看一下训练集中最常出现的词。这样能帮助理解模型看到的特征是什么。

In [ ]:
vocab = vectorizer.get_feature_names_out()
counts = np.asarray(train_features.sum(axis=0)).ravel()
top_words = sorted(zip(vocab, counts), key=lambda item: item[1], reverse=True)[:20]

for word, count in top_words:
    print(f'{word}: {int(count)}')

## 7. 本地验证

Kaggle 原教程通常直接训练全部数据并提交。

为了学习，可以先切出一部分训练集作为验证集，看看模型大概表现。

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    train_features,
    train['sentiment'],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train['sentiment'],
)

valid_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
valid_model.fit(X_train, y_train)
valid_pred = valid_model.predict(X_valid)

print('Validation accuracy:', accuracy_score(y_valid, valid_pred))
print(classification_report(y_valid, valid_pred))

## 8. 用全部训练集训练最终模型

验证结束后，用全部带标签训练数据重新训练模型。

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
forest.fit(train_features, train['sentiment'])

print('Final model trained.')

## 9. 处理测试集

测试集必须使用同样的清洗函数。

这里要注意，用的是 `vectorizer.transform`，不是 `fit_transform`。因为测试集必须使用训练集学到的同一套词表。

In [ ]:
clean_test_reviews = []

for i, review in enumerate(test['review'], start=1):
    if i % 1000 == 0:
        print(f'Cleaned {i} / {len(test)} test reviews')
    clean_test_reviews.append(review_to_words(review))

test_features = vectorizer.transform(clean_test_reviews)
print('Test feature matrix shape:', test_features.shape)

## 10. 生成 Kaggle 提交文件

模型会输出 `0` 或 `1`。

`1` 表示正面评论。  
`0` 表示负面评论。

在 Kaggle Notebook 里，输出文件会保存到 `/kaggle/working/Bag_of_Words_model.csv`。运行完成后可以在右侧 Output 区域下载。

In [ ]:
result = forest.predict(test_features)

submission = pd.DataFrame({
    'id': test['id'],
    'sentiment': result.astype(int),
})

output_path = Path('/kaggle/working/Bag_of_Words_model.csv')
if not output_path.parent.exists():
    output_path = Path('Bag_of_Words_model.csv')

submission.to_csv(output_path, index=False, quoting=csv.QUOTE_NONE)

print('Saved submission file:', output_path)
display(submission.head())

## 11. 总结

这个 Notebook 展示的是早期 NLP 的典型流程。

文本本身不能直接给机器学习模型使用，所以先清洗文本，再用 Bag-of-Words 转成词频向量。

随机森林会从训练数据里学习哪些词频模式更像正面评论，哪些更像负面评论。

这种方法简单、可解释、不需要 GPU，但它不真正理解语义，也不太考虑词序。后续 Word2Vec、Embedding、Transformer 和大模型，都是在进一步改进“如何表示文本”这个问题。